# Crimson Circle Library Scraper

This notebook downloads all shoud transcripts from [crimsoncircle.com/library](https://www.crimsoncircle.com/library) and gives you a `.zip` file at the end.

## How to use

1. Click **Runtime > Run all** (or press `Ctrl+F9`)
2. Wait for it to finish (may take 30-60 minutes for the full library)
3. A `.zip` file will automatically download to your computer when done

That's it! No setup, no installs, nothing.

---

## Step 1 - Install dependencies
This installs Playwright (a headless browser) and other tools. Takes ~1-2 minutes.

In [ ]:
!pip install -q playwright beautifulsoup4 html2text playwright-stealth
!playwright install chromium
!playwright install-deps chromium
print("\n--- All dependencies installed! ---")

## Step 2 - Configuration

You can adjust settings here if needed. The defaults should work fine.

In [ ]:
# ============================================================
# CONFIGURATION - adjust these if needed
# ============================================================

BASE_URL = "https://www.crimsoncircle.com"
LIBRARY_URL = f"{BASE_URL}/library"

# Output folder (inside this Colab session)
OUTPUT_DIR = "/content/crimson-circle-library"

# Delays (seconds) - be polite to the server
DELAY_BETWEEN_SERIES = 3
DELAY_BETWEEN_SHOUDS = 2
PAGE_LOAD_TIMEOUT = 30000  # milliseconds

# Retry on failure
MAX_RETRIES = 3
RETRY_DELAY = 5  # seconds

# Set to a string to only scrape one series (see Step 3 output for names)
SINGLE_SERIES = None  # e.g. "illumination-series"

print("Configuration loaded.")

## Step 3 - Discover the library structure

This visits the library page and lists every series + shoud it finds.  
Review the output to make sure it looks right before running the full download.

In [ ]:
import asyncio
import os
import re
import json
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeout
from playwright_stealth import stealth_async
import html2text

# ============================================================
# Helper functions
# ============================================================

EXCLUDE_URL_PATTERNS = [
    r"/library/?$",
    r"/login", r"/register", r"/cart", r"/account", r"/search",
    r"^#$", r"javascript:", r"mailto:",
]

def is_excluded(url: str) -> bool:
    for pattern in EXCLUDE_URL_PATTERNS:
        if re.search(pattern, url):
            return True
    return False

def slug_from_url(url: str) -> str:
    path = urlparse(url).path.rstrip("/")
    return path.split("/")[-1] if "/" in path else path

def html_to_clean_text(html_content: str) -> str:
    converter = html2text.HTML2Text()
    converter.ignore_links = True
    converter.ignore_images = True
    converter.ignore_emphasis = False
    converter.body_width = 0
    converter.unicode_snob = True
    text = converter.handle(html_content)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

async def wait_for_cloudflare(page, timeout=30):
    """Wait for Cloudflare challenge to resolve. Returns True if passed."""
    for _ in range(timeout):
        content = await page.content()
        # Cloudflare challenge pages have very little content
        # and contain cloudflare.com links or challenge scripts
        if "cloudflare" not in content.lower() or len(content) > 10000:
            # Check we have real page content (more than just CF challenge)
            soup = BeautifulSoup(content, "html.parser")
            links = soup.select("a[href]")
            if len(links) > 3:
                return True
        await asyncio.sleep(1)
    return False

async def safe_goto(page, url, retries=MAX_RETRIES):
    """Navigate to URL and wait for Cloudflare challenge to pass."""
    for attempt in range(retries):
        try:
            await page.goto(url, wait_until="domcontentloaded")
            passed = await wait_for_cloudflare(page)
            if passed:
                return True
            print(f"  Cloudflare challenge did not resolve (attempt {attempt+1}/{retries})")
            if attempt < retries - 1:
                await asyncio.sleep(5)
        except PlaywrightTimeout:
            print(f"  Timeout loading {url} (attempt {attempt+1}/{retries})")
            if attempt < retries - 1:
                await asyncio.sleep(5)
    return False

# ============================================================
# Launch STEALTH browser and discover series
# ============================================================

print("Launching stealth browser...")
pw = await async_playwright().start()
browser = await pw.chromium.launch(
    headless=True,
    args=[
        "--disable-blink-features=AutomationControlled",
        "--no-sandbox",
    ],
)
context = await browser.new_context(
    user_agent=(
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    viewport={"width": 1280, "height": 720},
    locale="en-US",
    timezone_id="America/New_York",
)
page = await context.new_page()

# Apply stealth patches to avoid bot detection
await stealth_async(page)

page.set_default_timeout(PAGE_LOAD_TIMEOUT)
print("Stealth browser ready.\n")

# === First visit: library page (establish Cloudflare clearance) ===
print(f"Visiting {LIBRARY_URL} ...")
ok = await safe_goto(page, LIBRARY_URL)
if not ok:
    print("ERROR: Could not get past Cloudflare on the library page.")
    print("Try running the cell again - sometimes it takes a second attempt.")

html = await page.content()
soup = BeautifulSoup(html, "html.parser")

all_series = []
seen = set()
for a in soup.select('a[href*="/library/"]'):
    href = a.get("href", "")
    if not href:
        continue
    full_url = urljoin(BASE_URL, href)
    if "/library/" not in full_url or is_excluded(full_url):
        continue
    path = urlparse(full_url).path.rstrip("/")
    parts = [p for p in path.split("/") if p]
    if len(parts) != 2 or parts[0] != "library":
        continue
    if full_url in seen:
        continue
    seen.add(full_url)
    name = a.get_text(strip=True) or slug_from_url(full_url)
    all_series.append({"name": name, "slug": parts[1], "url": full_url})

print(f"\nFound {len(all_series)} series:\n")
for i, s in enumerate(all_series, 1):
    print(f"  {i:2d}. {s['name']}")
    print(f"      {s['url']}")

### Discover shouds in each series

This visits each series page and lists the individual shoud pages.

**Note:** The site uses hrefs like `/{series-slug}/{shoud-slug}` (without `/library/`),  
but the actual pages live at `/library/{series-slug}/{shoud-slug}`.

In [ ]:
# ============================================================
# Discover shouds within each series
# ============================================================

library_structure = []
total_shouds = 0

for i, series in enumerate(all_series, 1):
    print(f"\n[{i}/{len(all_series)}] {series['name']}")
    print(f"  Visiting {series['url']} ...")

    ok = await safe_goto(page, series["url"])
    if not ok:
        print(f"  BLOCKED by Cloudflare - skipping series")
        library_structure.append({"series": series, "shouds": []})
        await asyncio.sleep(DELAY_BETWEEN_SERIES)
        continue

    html = await page.content()
    soup = BeautifulSoup(html, "html.parser")

    shouds = []
    seen_shoud_urls = set()
    series_slug = series["slug"]

    # Find all links on the page that contain the series slug
    for a in soup.select("a[href]"):
        href = a.get("href", "")
        if not href or series_slug not in href:
            continue

        # Normalize: strip leading/trailing slashes and protocol/domain
        href_clean = href.strip("/")
        if href_clean.startswith("http"):
            href_clean = urlparse(href_clean).path.strip("/")
        parts = href_clean.split("/")

        # Accept: "series-slug/shoud-slug" or "library/series-slug/shoud-slug"
        shoud_slug = None
        if len(parts) == 2 and parts[0] == series_slug:
            shoud_slug = parts[1]
        elif len(parts) == 3 and parts[0] == "library" and parts[1] == series_slug:
            shoud_slug = parts[2]

        if not shoud_slug:
            continue

        full_url = f"{BASE_URL}/library/{series_slug}/{shoud_slug}"
        if full_url in seen_shoud_urls:
            continue
        seen_shoud_urls.add(full_url)

        name = a.get_text(strip=True) or shoud_slug
        shouds.append({"name": name, "slug": shoud_slug, "url": full_url})

    library_structure.append({"series": series, "shouds": shouds})
    total_shouds += len(shouds)
    print(f"  Found {len(shouds)} shouds")
    for s in shouds:
        print(f"    - {s['name']}")

    await asyncio.sleep(DELAY_BETWEEN_SERIES)

print(f"\n{'='*60}")
print(f"TOTAL: {len(all_series)} series, {total_shouds} shouds")
print(f"{'='*60}")

## Step 4 - Download all transcripts

This is the main scraping step. It visits every shoud page, extracts the transcript, and saves it as a `.txt` file.

**This will take a while** (~30-60 min for the full library). You can watch progress below.  
If it gets interrupted, just re-run this cell - it skips already downloaded files.

In [ ]:
# ============================================================
# Download all transcripts
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

downloaded = 0
skipped = 0
failed = 0
failed_list = []

for entry in library_structure:
    series = entry["series"]
    shouds = entry["shouds"]

    if SINGLE_SERIES and SINGLE_SERIES.lower() not in series["slug"].lower():
        continue

    series_dir = os.path.join(OUTPUT_DIR, series["slug"])
    os.makedirs(series_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"Series: {series['name']} ({len(shouds)} shouds)")
    print(f"{'='*60}")

    for j, shoud in enumerate(shouds, 1):
        filepath = os.path.join(series_dir, f"{shoud['slug']}.txt")

        if os.path.exists(filepath) and os.path.getsize(filepath) > 500:
            print(f"  [{j}/{len(shouds)}] SKIP (exists): {shoud['name']}")
            skipped += 1
            continue

        print(f"  [{j}/{len(shouds)}] Downloading: {shoud['name']}")

        try:
            ok = await safe_goto(page, shoud["url"])
            if not ok:
                print(f"           BLOCKED by Cloudflare")
                failed += 1
                failed_list.append(shoud["url"])
                await asyncio.sleep(DELAY_BETWEEN_SHOUDS)
                continue

            # Scroll to trigger lazy loading
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            await page.wait_for_timeout(1500)

            html = await page.content()
            soup = BeautifulSoup(html, "html.parser")

            # Extract transcript from the known container
            best_text = ""
            transcript_el = soup.select_one('#transcript-ShoudTranscript')
            if transcript_el:
                for tag in transcript_el.find_all(["nav", "header", "footer", "script", "style", "noscript"]):
                    tag.decompose()
                best_text = html_to_clean_text(str(transcript_el))

            if best_text and len(best_text) > 200:
                header = (
                    f"Title: {shoud['name']}\n"
                    f"Series: {series['name']}\n"
                    f"Source: {shoud['url']}\n"
                    f"{'='*60}\n\n"
                )
                with open(filepath, "w", encoding="utf-8") as f:
                    f.write(header + best_text)
                downloaded += 1
                print(f"           Saved ({len(best_text):,} chars)")
            else:
                failed += 1
                failed_list.append(shoud["url"])
                print(f"           WARNING: No transcript found ({len(best_text)} chars)")

        except Exception as e:
            failed += 1
            failed_list.append(shoud["url"])
            print(f"           ERROR: {e}")

        await asyncio.sleep(DELAY_BETWEEN_SHOUDS)

print(f"\n{'='*60}")
print(f"DONE!")
print(f"  Downloaded: {downloaded}")
print(f"  Skipped (already existed): {skipped}")
print(f"  Failed: {failed}")
if failed_list:
    print(f"\nFailed URLs:")
    for url in failed_list:
        print(f"  - {url}")
print(f"{'='*60}")

## Step 5 - Close the browser

In [ ]:
await page.close()
await context.close()
await browser.close()
await pw.stop()
print("Browser closed.")

## Step 6 - Preview what was downloaded

In [ ]:
# Show the folder structure
print("Downloaded files:\n")
total_files = 0
total_size = 0
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, "").count(os.sep)
    indent = "  " * level
    folder_name = os.path.basename(root)
    txt_files = [f for f in files if f.endswith(".txt")]
    if txt_files:
        print(f"{indent}{folder_name}/ ({len(txt_files)} files)")
        for f in sorted(txt_files):
            fpath = os.path.join(root, f)
            size = os.path.getsize(fpath)
            total_size += size
            total_files += 1
            print(f"{indent}  {f} ({size:,} bytes)")

print(f"\nTotal: {total_files} files, {total_size:,} bytes ({total_size/1024/1024:.1f} MB)")

## Step 7 - Download as ZIP

This creates a `.zip` file and triggers a download to your computer.

In [ ]:
import shutil

zip_path = "/content/crimson-circle-library"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)

zip_file = zip_path + ".zip"
zip_size = os.path.getsize(zip_file)
print(f"ZIP created: {zip_file} ({zip_size:,} bytes / {zip_size/1024/1024:.1f} MB)")

# Auto-download in Colab
try:
    from google.colab import files
    files.download(zip_file)
    print("\nDownload started! Check your browser's downloads.")
except ImportError:
    print(f"\nNot running in Colab. Grab the file manually at: {zip_file}")